# Generative AI Application Deployment and Monitoring

## Overview

- Load a model previously created to perform text summarization

- Create a Databricks endpoint to serve the model

- Run inference and compute some metrics in near real time

- Configure real-time performance monitoring in Databricks

## Initial setup

In [0]:
%sql
use catalog `studies`;
use schema `databricks-dev`;

## Load a model previously created

In [0]:
database_name = spark.sql('SELECT current_database()').collect()[0][0]
catalog_name = spark.sql('SELECT current_catalog()').collect()[0][0]
schema_name = spark.sql('SELECT current_schema()').collect()[0][0]

print(f'database_name: {database_name}')
print(f'catalog_name: {catalog_name}')
print(f'schema_name: {schema_name}')

database_name: `databricks-dev`

catalog_name: studies

schema_name: `databricks-dev`

In [0]:
import mlflow

model_name = f'{catalog_name}.{schema_name}.model_databricks_studies'

client = mlflow.MlflowClient()
mlflow.set_registry_uri(database_name)

In [0]:
model_name = 'summarizer_test'
versions = client.get_latest_versions(name=model_name)
current_model_version_candidates = []
for v in versions:
    print(f'Version: {v.version}')
    print(f'Stage: {v.current_stage}')
    print(f'Run ID: {v.run_id}')
    if v.run_id:
        current_model_version_candidates.append({'version': v.version, 'stage': v.current_stage})

/home/spark-4cb6e7d6-8fa7-4e28-8150-4b/.ipykernel/3431/command-8453349636839616-3266932822:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  versions = client.get_latest_versions(name=model_name)


Version: 5

Stage: None

Run ID: 83098a4441e24f6ca879603aa105d957

In [0]:
current_model_version_candidates

[{'version': 5, 'stage': 'None'}]

In [0]:
current_model_version = current_model_version_candidates[-1]['version']
current_model_stage = current_model_version_candidates[-1]['stage']
print(f'current_model_version: {current_model_version}, current_model_stage: {current_model_stage}')

latest_model = mlflow.pyfunc.load_model(
	model_uri=f'models:/{model_name}/{current_model_version}',
)
latest_model

current_model_version: 5, current_model_stage: None

mlflow.pyfunc.loaded_model:
  artifact_path: dbfs:/databricks/mlflow-tracking/3717431272157322/logged_models/m-eadbb9c4c3d5498aa1e2bceec18e833a/artifacts
  flavor: mlflow.transformers
  run_id: 83098a4441e24f6ca879603aa105d957

## Model serving using the SDK API

### Setup secrets

<br/>

```shell
# terminal
databricks secrets list-scopes
databricks secrets create-scope genai_deploy
databricks secrets put-secret genai_deploy GENAI_API_KEY

# 15b0c646608d43af9d74e69e589140f5
---

databricks secrets put-secret --json '{ "scope": "genai_deploy", "key": "depl_host", "string_value": "<host-name>"}
databricks secrets put-secret --json '{ "scope": "genai_deploy", "key": "depl_token", "string_value": "<token-value>"}

```

In [0]:
from databricks.sdk.service.serving import EndpointCoreConfigInput 

endpoint_config_dict = {
	'served_models': [{
		'model_name': model_name,
		'model_version': latest_model,
		'scale_to_zero_enabled': True,
		'workload_size': 'Small',
		'enviroment_vars': {
			'DATABRICKS_TOKEN': '{{secrets/latest_model/depl_token}}',
			'DATABRICKS_HOST': '{{secrets/latest_model/depl_host}}'
		}
	}],
	'auto_capture_config': {
		'catalog_name': catalog_name,
		'schema_name': schema_name,
		'table_name_prefix': 'summarize_task_realtime'
	}
}

endpoint_config = EndpointCoreConfigInput.from_dict(endpoint_config_dict)

In [0]:
endpoint_config

EndpointCoreConfigInput(auto_capture_config=AutoCaptureConfigInput(catalog_name='studies', enabled=None, schema_name='`databricks-dev`', table_name_prefix='summarize_task_realtime'), name=None, served_entities=[], served_models=[ServedModelInput(scale_to_zero_enabled=True, model_name='summarizer_test', model_version=mlflow.pyfunc.loaded_model:
  artifact_path: dbfs:/databricks/mlflow-tracking/3717431272157322/logged_models/m-eadbb9c4c3d5498aa1e2bceec18e833a/artifacts
  flavor: mlflow.transformers
  run_id: 83098a4441e24f6ca879603aa105d957
, environment_vars=None, instance_profile_arn=None, max_provisioned_throughput=None, min_provisioned_throughput=None, name=None, workload_size=<ServedModelInputWorkloadSize.SMALL: 'Small'>, workload_type=None)], traffic_config=None)

### Deploy

In [0]:
# from databricks import WorkspaceClient

# workspace_client = WorkspaceClient()
# print([endpoint for endpoint in workspace_client.list()])

# endpoint_name = ...
# db_host = ...

# endpoint_url = f'{db_host}/ml/endpoints/{endpoint_name}'

# workspace_client.serving_endpoints.update_config_and_wait(name=endpoint_name, served_models=edpoint_config.served_models)

## Model serving via UI
  - Experiments > Select a experiment (experiment_name) > Choose a Run name > Click on `Register model`
  - Fill the fields to register the model:
    - Notes:
      - Model name valid format: `<catalog_name>.<schema_name>.<name_model_choosen>`
      - Estimated time to create the endpoint: ~ 8 min.

## Run inferences

In [0]:
from rich import print
import json
import pandas as pd

JSON_WRITER_SUMMARIES_PATH = '/Volumes/studies/databricks-dev/data_json/writer_summaries.json'

In [0]:
def load_json(json_name):
    with open(json_name, 'r', encoding='utf-8') as f:
        data = json.load(f)
        return data
    
def pandas_to_databricks_table(table_name: str, pandas_df: pd.DataFrame) -> None:
    spark_df = spark.createDataFrame(pandas_df)
    spark_df.write.format('delta').mode('overwrite').saveAsTable(table_name)
    print(spark_df.count)
    display(spark_df.show(5))

In [0]:
df_writer_summaries = pd.DataFrame(load_json(JSON_WRITER_SUMMARIES_PATH))
df_writer_summaries.rename(columns={'summary': 'human_summary'}, inplace=True)
df_writer_summaries['len_article'] = df_writer_summaries.article.apply(lambda x: len(x))
print(df_writer_summaries.shape)
df_writer_summaries.head()

(302, 4)

,article_id,article,human_summary,len_article
0,0adb86356834452298d180104ff54179,Nick Scholfield is lined up to ride Spring Hee...,Nick Schofield is riding Spring Heeled in the ...,2650
1,0adb86356834452298d180104ff54179,Nick Scholfield is lined up to ride Spring Hee...,Preparation is taking place for a horse ridin...,2650
2,0adb86356834452298d180104ff54179,Nick Scholfield is lined up to ride Spring Hee...,Nick Scholfield will travel to Ireland and is ...,2650
3,b3168ab4857d4190ac3b2eb46d096f81,Dr Mehmet Oz's fellow faculty members at Colum...,Faculty at Columbia University have written an...,7792
4,b3168ab4857d4190ac3b2eb46d096f81,Dr Mehmet Oz's fellow faculty members at Colum...,"Celebrity doctor, Mehmet Oz, is being attacked...",7792


In [0]:
df_sample = df_writer_summaries[df_writer_summaries.len_article < 1000]
df_sample

,article_id,article,human_summary,len_article
74,fbb2c1430c3548d4819efec14ba22765,(CNN) -- A high-speed passenger train left its...,A high-speed passenger train derailed on Frida...,661
100,2e5837f2f9e440d0b4bd6268f874dd17,"BAGHDAD, Iraq (CNN) -- Six gay men were shot d...",Six gay men were shot dead by members of their...,768
255,094ba204fcad42d9b13a888c50e0d61d,Edinburgh's winter festivals generated more th...,Edinburgh's Christmas and Hogmanay festivals g...,787
260,d5974b0899c84a7eac8d8c681c8f697a,DNA in dog mess could be used to catch owners ...,DNA in dog poop can now be used to catch owner...,860
272,73a12c43e31346bf8c563bf8050d0b9b,WASHINGTON (CNN) -- The U.S. Navy has charged ...,The US Navy has charged six guards with assaul...,888
283,78bb8d70ea224917afcd41e9fd8539ef,Golfers at Dundee's public courses have been b...,Golfers at Dundee's public courses can no long...,810
298,2550949dbb23499a837b10f16ebd56f1,"Rare drawings by Leonardo da Vinci, which are ...",Rare drawings by Leonardo da Vinci have gone o...,761
299,b14baa3916194510a74f074f415ce10a,"BANGKOK, Thailand (CNN) -- A Bangkok Airways p...",A Bangkok Airways plane crashed killing the pi...,863


In [0]:
one_sample = df_sample.iloc[0].to_dict()
print(f'** Length of Article:** {one_sample['len_article']}')
print(f'**Article:**\n{one_sample['article']}')
print(f'**Human Summary:**\n{one_sample['human_summary']}')

** Length of Article:** 661

**Article:**
(CNN) -- A high-speed passenger train left its tracks on the outskirts of Split, Croatia, Friday, killing at least 
six people and injuring 45, according to Croatian police. The high-speed train derailed on the outskirts of Split, 
Croatia, about noon on Friday. The train was on its way from the Croatian capital, Zagreb, when it derailed about 
20 kilometers (12 miles) from it's destination of Split about noon, said Marina Kraljevic-Gudelj, a spokeswoman for
police in Split. "This is a huge tragedy, so there is no place for speculation," she said. Police had launched an 
investigation into the cause of the crash. CNN's Per Nyberg contributed to this report.

**Human Summary:**
A high-speed passenger train derailed on Friday, killing six and injuring 45. Police have launched an investigation
into the cause of the crash which occurred on the outskirts of Split, Croatia.

### Inference using SDK API

In [0]:
from databricks.sdk import WorkspaceClient

workspace_client = WorkspaceClient()

In [0]:
endpoint_uri = 'https://dbc-34ac7b3c-7a54.cloud.databricks.com/serving-endpoints/studies/invocations'

In [0]:
idx = 5
test_sample = df_sample.iloc[idx].to_dict()['article']
test_sample

'Golfers at Dundee\'s public courses have been banned from bringing their dogs with them after complaints from fellow players and staff. It follows reports of dog fouling and damage at the Camperdown and Caird Park courses. Dogs can still be walked across the courses but not if owners are playing a round of the game at the time. A spokesman for Leisure and Culture Dundee said the rules were changed on 20 April. He said: "This change reflects the concerns of many players and staff about dog fouling and damage being caused to the courses, particularly greens and bunkers. "The new management rules, which do not affect the Right to Roam legislation, are clearly signed at the courses and on the Leisure and Culture Dundee website. "Most golf courses in Scotland do not allow players to bring dogs with them."'

In [0]:

res = workspace_client.serving_endpoints.query(
    'studies', 
    inputs=[test_sample]
)

print(f'Article: {test_sample}')
print('Summary (model): ', res.predictions)

Article: Golfers at Dundee's public courses have been banned from bringing their dogs with them after complaints 
from fellow players and staff. It follows reports of dog fouling and damage at the Camperdown and Caird Park 
courses. Dogs can still be walked across the courses but not if owners are playing a round of the game at the time.
A spokesman for Leisure and Culture Dundee said the rules were changed on 20 April. He said: "This change reflects 
the concerns of many players and staff about dog fouling and damage being caused to the courses, particularly 
greens and bunkers. "The new management rules, which do not affect the Right to Roam legislation, are clearly 
signed at the courses and on the Leisure and Culture Dundee website. "Most golf courses in Scotland do not allow 
players to bring dogs with them."

Summary (model): 
[
    'Golfers in Dundee have been banned from bringing their dogs with them after a change to rules relating to the 
right to Roam.'
]

### Inference with `ai_query`

In [0]:
%sql
SELECT ai_query(
'studies',
request => named_struct(
'article', 
'{test_sample}'
))

"ai_query('studies',request=>named_struct('article','{test_sample}'))"
test_sample - a sample of a test in the UK - is being used to test the results of the test.


### Inference with MLflow

In [0]:
from mlflow.deployments import get_deploy_client

deploy_client = get_deploy_client('databricks')
res_mlflow = deploy_client.predict(
	endpoint='studies',
 
	inputs={'inputs': [test_sample]}
)

print(res_mlflow.predictions)

[
    'Golfers in Dundee have been banned from bringing their dogs with them after a change to rules relating to the 
right to Roam.'
]

## Calculate metrics

In [0]:
N_MAX = 3
df_metrics = df_sample.head(N_MAX)

In [0]:
pred_summaries = latest_model.predict(df_metrics.article)
print(f'\nArticles:\n{df_metrics.article.tolist()}')
print(f'\nPredicted Summaries:\n{pred_summaries}')
print(pred_summaries)

Your max_length is set to 200, but your input_length is only 167. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=83)
Your max_length is set to 200, but your input_length is only 178. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=89)
Your max_length is set to 200, but your input_length is only 177. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=88)


Articles:
['(CNN) -- A high-speed passenger train left its tracks on the outskirts of Split, Croatia, Friday, killing at 
least six people and injuring 45, according to Croatian police. The high-speed train derailed on the outskirts of 
Split, Croatia, about noon on Friday. The train was on its way from the Croatian capital, Zagreb, when it derailed 
about 20 kilometers (12 miles) from it\'s destination of Split about noon, said Marina Kraljevic-Gudelj, a 
spokeswoman for police in Split. "This is a huge tragedy, so there is no place for speculation," she said. Police 
had launched an investigation into the cause of the crash. CNN\'s Per Nyberg contributed to this report.', 
"BAGHDAD, Iraq (CNN) -- Six gay men were shot dead by members of their tribe in two separate incidents in the past 
10 days, an official with Iraq's Interior ministry said. In the most recent attack, two men were killed Thursday in
Sadr City area of Baghdad after they were disowned by relatives, the official said. The shootings came after a 
tribal meeting was held and the members decided to go after the victims. On March 26, four additional men were 
fatally shot in the same city, the official said, adding that the victims had also been disowned by their 
relatives. The official declined to be identified because he is not authorized to speak to the media. Witnesses 
told CNN that a Sadr City cafe, which was a popular gathering spot for gays, was also set on fire.", 'Edinburgh\'s 
winter festivals generated more than £241m for the city, according to organisers. Almost one million people visited
the city during the six-week festival period over Christmas and Hogmanay. Organisers said almost 890,000 people 
visited the Edinburgh\'s Christmas events in 2014/15, contributing £199.5m to the local economy. The three-day 
Hogmanay celebrations attracted more than 150,000 people, creating an economic impact of £41.8m. Charlie Wood, 
Edinburgh\'s Christmas festival director, said: "This is great news for Edinburgh. The revenue generated does not 
go to the events themselves, the event organisers or to Edinburgh city council. "This is money, which is going to 
the businesses of Edinburgh, be it retail, accommodation, food, drink, shopping and entertainment."']

Predicted Summaries:
['A high-speed train has derailed in Croatia, killing at least six people and injuring more than a dozen others, 
police have said.', 'Six gay men have been shot dead in the Iraqi city of Baghdad, officials say, after a tribal 
meeting was held in the city.', "More than a million people visited Edinburgh's Christmas festivals in 2014/15, 
according to the city's organisers and the city council."]

[
    'A high-speed train has derailed in Croatia, killing at least six people and injuring more than a dozen others,
police have said.',
    'Six gay men have been shot dead in the Iraqi city of Baghdad, officials say, after a tribal meeting was held 
in the city.',
    "More than a million people visited Edinburgh's Christmas festivals in 2014/15, according to the city's 
organisers and the city council."
]

In [0]:
# %pip install textstat evaluate

In [0]:
from pyspark.sql.functions import pandas_udf
import textstat

@pandas_udf('double')
def lexicon_count(texts):
	return pd.Series([textstat.lexicon_count(text, removepunct=True) for text in texts])

@pandas_udf('double')
def reading_time(texts):
	return pd.Series([textstat.reading_time(text, ms_per_char=14.69) for text in texts])

@pandas_udf('double')
def automated_readability_index(texts):
	return pd.Series([textstat.automated_readability_index(text) for text in texts])

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

def calculate_metrics(df, cols):
    for col_name in cols:
	    df = (
		    df.withColumn(f'lexicon_count_{col_name}', lexicon_count(F.col(col_name)))
      		  .withColumn(f'reading_time_{col_name}', reading_time(F.col(col_name)))
        	  .withColumn(f'automated_readability_index{col_name}', automated_readability_index(F.col(col_name)))
		)
    df = df.withColumn('processing_timestamp', F.current_timestamp())
    return df


In [0]:
df_metrics.loc[:, 'summary_pred'] = pred_summaries
df_metrics.loc[:, 'len_summary_pred'] = df_metrics.summary_pred.apply(lambda x: len(x))
df_metrics

,article_id,article,human_summary,len_article,summary_pred,len_summary_pred
74,fbb2c1430c3548d4819efec14ba22765,(CNN) -- A high-speed passenger train left its...,A high-speed passenger train derailed on Frida...,661,"A high-speed train has derailed in Croatia, ki...",128
100,2e5837f2f9e440d0b4bd6268f874dd17,"BAGHDAD, Iraq (CNN) -- Six gay men were shot d...",Six gay men were shot dead by members of their...,768,Six gay men have been shot dead in the Iraqi c...,121
255,094ba204fcad42d9b13a888c50e0d61d,Edinburgh's winter festivals generated more th...,Edinburgh's Christmas and Hogmanay festivals g...,787,More than a million people visited Edinburgh's...,135


In [0]:
sample_spark = spark.createDataFrame(df_metrics) 
sample_spark.show()

+--------------------+--------------------+--------------------+-----------+--------------------+----------------+
|          article_id|             article|       human_summary|len_article|        summary_pred|len_summary_pred|
+--------------------+--------------------+--------------------+-----------+--------------------+----------------+
|fbb2c1430c3548d48...|(CNN) -- A high-s...|A high-speed pass...|        661|A high-speed trai...|             128|
|2e5837f2f9e440d0b...|BAGHDAD, Iraq (CN...|Six gay men were ...|        768|Six gay men have ...|             121|
|094ba204fcad42d9b...|Edinburgh's winte...|Edinburgh's Chris...|        787|More than a milli...|             135|
+--------------------+--------------------+--------------------+-----------+--------------------+----------------+



In [0]:
res_metrics = calculate_metrics(df=sample_spark, cols=['article', 'summary_pred'])
res_metrics.show()

+--------------------+--------------------+--------------------+-----------+--------------------+----------------+---------------------+--------------------+----------------------------------+--------------------------+-------------------------+---------------------------------------+--------------------+
|          article_id|             article|       human_summary|len_article|        summary_pred|len_summary_pred|lexicon_count_article|reading_time_article|automated_readability_indexarticle|lexicon_count_summary_pred|reading_time_summary_pred|automated_readability_indexsummary_pred|processing_timestamp|
+--------------------+--------------------+--------------------+-----------+--------------------+----------------+---------------------+--------------------+----------------------------------+--------------------------+-------------------------+---------------------------------------+--------------------+
|fbb2c1430c3548d48...|(CNN) -- A high-s...|A high-speed pass...|        661|A h

In [0]:
df_res_metrics = res_metrics.toPandas()
df_res_metrics

,article_id,article,human_summary,len_article,summary_pred,len_summary_pred,lexicon_count_article,reading_time_article,automated_readability_indexarticle,lexicon_count_summary_pred,reading_time_summary_pred,automated_readability_indexsummary_pred,processing_timestamp
0,fbb2c1430c3548d4819efec14ba22765,(CNN) -- A high-speed passenger train left its...,A high-speed passenger train derailed on Frida...,661,"A high-speed train has derailed in Croatia, ki...",128,108.0,8.12357,11.465688,22.0,1.57183,12.477727,2026-02-11 19:08:31.512970
1,2e5837f2f9e440d0b4bd6268f874dd17,"BAGHDAD, Iraq (CNN) -- Six gay men were shot d...",Six gay men were shot dead by members of their...,768,Six gay men have been shot dead in the Iraqi c...,121,137.0,9.26939,11.522971,24.0,1.43962,9.802500,2026-02-11 19:08:31.512970
2,094ba204fcad42d9b13a888c50e0d61d,Edinburgh's winter festivals generated more th...,Edinburgh's Christmas and Hogmanay festivals g...,787,More than a million people visited Edinburgh's...,135,113.0,9.91575,13.767456,20.0,1.70404,15.888000,2026-02-11 19:08:31.512970


In [0]:
pandas_to_databricks_table(table_name='metrics_summarizer', pandas_df=df_res_metrics)

<bound method DataFrame.count of DataFrame[article_id: string, article: string, human_summary: string, len_article:
bigint, summary_pred: string, len_summary_pred: bigint, lexicon_count_article: double, reading_time_article: 
double, automated_readability_indexarticle: double, lexicon_count_summary_pred: double, reading_time_summary_pred: 
double, automated_readability_indexsummary_pred: double, processing_timestamp: timestamp]>

+--------------------+--------------------+--------------------+-----------+--------------------+----------------+---------------------+--------------------+----------------------------------+--------------------------+-------------------------+---------------------------------------+--------------------+
|          article_id|             article|       human_summary|len_article|        summary_pred|len_summary_pred|lexicon_count_article|reading_time_article|automated_readability_indexarticle|lexicon_count_summary_pred|reading_time_summary_pred|automated_readability_indexsummary_pred|processing_timestamp|
+--------------------+--------------------+--------------------+-----------+--------------------+----------------+---------------------+--------------------+----------------------------------+--------------------------+-------------------------+---------------------------------------+--------------------+
|fbb2c1430c3548d48...|(CNN) -- A high-s...|A high-speed pass...|        661|A h

<img src="./imgs/metrics_profile_table.png">